In [ ]:
'''
Done: whole corpus parsed (only 'text' and 'conclusion' are saved') to
the single .csv file. Whole corpus is then indexed and 
evaluated via 'map' and 'ndcg'.
'''

In [ ]:
!pip install ijson

In [ ]:
import ijson
import re
data_set = set()
id_text = {}
filenames = ['debateorg', 'debatepedia', 'debatewise', 'idebate', 'parliamentary']
for filename in filenames:
  path = '/content/drive/MyDrive/Bachelor_webis/' + filename + '.json'
  with open(path, 'r', encoding='utf-8') as f:
      objects = ijson.items(f, 'arguments.item')
      for i, o in enumerate(objects):
        # text = 
        id_text[i] = o['id'], o["premises"][0]["text"] + o["conclusion"] 

In [ ]:
import pandas as pd
df = pd.DataFrame.from_dict(id_text).transpose()
df = df.rename(columns={0: "docno", 1: "text"})


In [ ]:
df.to_csv(index=False, path_or_buf='/content/drive/MyDrive/workspace/corpus.csv')

In [ ]:
!pip install python-terrier
import pyterrier as pt
if not pt.started():
  pt.init()

In [ ]:
pd_indexer = pt.DFIndexer("./pd_index")
indexref = pd_indexer.index(df["text"], df["docno"])

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/content/drive/MyDrive/workspace/topics.xml')
data = tree.getroot()
titles = []
title_ids = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    title_ids.append(num)


In [ ]:
topics = pd.DataFrame(titles,columns = ['query'])
topics['qid'] = topics.index + 51
columns_titles = ["qid","query"]
topics=topics.reindex(columns=columns_titles)

In [ ]:
index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)
qrels_path=("/content/drive/MyDrive/workspace/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

In [ ]:
res

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_5"])
eval